# Qwen GGUF → Signed .axm (Google Drive)

Paste a Google Drive share link to your Qwen GGUF, add Axiom training data,
and pack everything into a signed `.axm` governance container for edge/local deployment.

**Steps:**
1. `Cell 1` — install deps + GPU check
2. `Cell 2` — clone axiom + AXIOM_MASTER_KEY + llama.cpp
3. `Cell 3` — ⚙️ **edit here**: paste your Google Drive link + model info
4. `Cell 4` — download GGUF from Google Drive
5. `Cell 5` — prepare Axiom training data
6. `Cell 6` — pack GGUF + training data → signed .axm
7. `Cell 7` — verify HMAC proof chain
8. `Cell 8` — summary + download + inference commands

> **Note:** `axiom_axm` is NOT a pip package — it lives inside the cloned axiom repo.
> Cell 2 handles the clone automatically. Do not pip install it.

In [ ]:
# Cell 1 — GPU check + install dependencies
import subprocess, sys

try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10)
    if gpu.returncode == 0:
        for line in gpu.stdout.strip().splitlines():
            print(f"  GPU: {line}")
    else:
        print("  ⚠ No GPU detected — packing works on CPU; inference will be slow")
except Exception:
    print("  GPU: unknown")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "gdown", "huggingface_hub"], check=True)
print("✓ ready")

In [ ]:
# Cell 2 — clone axiom repo + AXIOM_MASTER_KEY + pre-built llama.cpp
import io, json, os, secrets, subprocess, sys, urllib.request, zipfile
from pathlib import Path

AXIOM_DIR  = Path("/content/axiom")       # RunPod: /workspace/axiom
OUTPUT_DIR = Path("/content/axm_output")  # RunPod: /workspace/axm_output
LLAMA_DIR  = Path("/content/llama.cpp")   # RunPod: /workspace/llama.cpp
BRANCH     = "claude/srd-prototype-benchmark-JRtv1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Clone axiom ────────────────────────────────────────────────────────────
if not AXIOM_DIR.is_dir():
    print("Cloning axiom repo...")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
        "https://github.com/orivael-dev/axiom.git", str(AXIOM_DIR)], check=True)
    print("✓ axiom cloned")
else:
    print(f"✓ axiom at {AXIOM_DIR}")
sys.path.insert(0, str(AXIOM_DIR))

# ── 2. AXIOM_MASTER_KEY (persistent across cell re-runs) ──────────────────────
KEY_FILE = OUTPUT_DIR / "axiom_master.key"
if os.environ.get("AXIOM_MASTER_KEY"):
    print("AXIOM_MASTER_KEY: from environment")
elif KEY_FILE.is_file():
    os.environ["AXIOM_MASTER_KEY"] = KEY_FILE.read_text().strip()
    print(f"AXIOM_MASTER_KEY: restored from {KEY_FILE}")
else:
    key = secrets.token_hex(32)
    os.environ["AXIOM_MASTER_KEY"] = key
    KEY_FILE.write_text(key)
    print(f"AXIOM_MASTER_KEY: generated → {KEY_FILE}")
    print("  ⚠ Copy this key to Google Drive — you need it to verify the AXM later:")
    print(f"  {key}")

# ── 3. Pre-built llama.cpp from GitHub releases ───────────────────────────────
def _setup_llamacpp(llama_dir: Path) -> Path:
    binary = llama_dir / "build" / "bin" / "llama-cli"
    if binary.is_file():
        print(f"✓ llama.cpp already at {binary}")
        return binary

    # Detect CUDA version
    cuda_tag = ""
    try:
        import torch
        cuda_ver = torch.version.cuda or ""
        parts = cuda_ver.split(".")
        if len(parts) >= 2:
            cuda_tag = f"cu{parts[0]}{parts[1]}"
    except ImportError:
        pass

    # Fetch latest GitHub release
    print("Fetching pre-built llama.cpp from GitHub...")
    try:
        req = urllib.request.Request(
            "https://api.github.com/repos/ggerganov/llama.cpp/releases/latest",
            headers={"User-Agent": "axiom-notebook/1.0"})
        with urllib.request.urlopen(req, timeout=20) as r:
            release = json.loads(r.read())
    except Exception as e:
        print(f"  GitHub API failed: {e}")
        release = {"assets": []}

    # Score assets: prefer exact CUDA match, then any ubuntu-x64
    def _score(a):
        n = a["name"]
        if "ubuntu-x64" not in n or not n.endswith(".zip"):
            return 0
        if cuda_tag and cuda_tag in n:
            return 3
        if "cuda" in n.lower():
            return 2
        return 1

    best = max(release.get("assets", []), key=_score, default=None)
    if best and _score(best) > 0:
        mb = best["size"] / 1024**2
        print(f"  Downloading {best['name']} ({mb:.0f} MB)...")
        try:
            build_bin = llama_dir / "build" / "bin"
            build_bin.mkdir(parents=True, exist_ok=True)
            with urllib.request.urlopen(best["browser_download_url"], timeout=300) as r:
                data = r.read()
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for member in z.namelist():
                    stem = Path(member).name
                    if stem in ("llama-cli", "llama-quantize"):
                        dest = build_bin / stem
                        dest.write_bytes(z.read(member))
                        dest.chmod(0o755)
                        print(f"  ✓ {stem}")
            if binary.is_file():
                print("✓ llama.cpp pre-built binary ready")
                return binary
        except Exception as e:
            print(f"  Pre-built download failed: {e}")

    # Fallback: build from source
    print("  Falling back to source build (~10-15 min)...")
    if not llama_dir.is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/ggerganov/llama.cpp.git", str(llama_dir)], check=True)
    build_dir = llama_dir / "build"
    build_dir.mkdir(exist_ok=True)
    subprocess.run(["cmake", "..", "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"],
        cwd=build_dir, check=True)
    subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j4"],
        cwd=build_dir, check=True)
    print("✓ llama.cpp built from source")
    return binary

LLAMA_BIN = _setup_llamacpp(LLAMA_DIR)
print(f"\n✓ All setup complete")
print(f"  axiom:      {AXIOM_DIR}")
print(f"  output:     {OUTPUT_DIR}")
print(f"  llama-cli:  {LLAMA_BIN}")

In [ ]:
# Cell 3 — ⚙️  CONFIGURE  (edit this cell)
#
# ── GGUF source ───────────────────────────────────────────────────────────────
# Paste your Google Drive share link, or a mounted Drive path.
# Right-click the .gguf file in Google Drive → Share → Copy link
#
# Share link:   https://drive.google.com/file/d/1aBcDeFg.../view?usp=sharing
# Mounted path: /content/drive/MyDrive/models/qwen2.5-coder-0.5b-q4_k_m.gguf

GDRIVE_LINK   = "PASTE_YOUR_GOOGLE_DRIVE_LINK_HERE"
GGUF_FILENAME = "qwen2.5-coder-0.5b-instruct-q4_k_m.gguf"  # local save name
MODEL_NAME    = "Qwen/Qwen2.5-Coder-0.5B-Instruct"          # for AXM header
AXM_NAME      = None   # None → auto from GGUF_FILENAME, or e.g. "qwen_coder.axm"

# ── Built-in Axiom training sets (from cloned repo) ───────────────────────────
# Set INCLUDE_TRAINING_DATA = False to skip all training data.
INCLUDE_TRAINING_DATA = True

TRAINING_SETS = [
    "axiom_metric_targeted",   # 4999 examples — behavioral governance training
    "train_qwen_chatml",       # 438  examples — Qwen ChatML format training
]

# ── Custom JSONL files (your own training data) ────────────────────────────────
# Add any number of entries. Three input formats are supported:
#
#   Google Drive share link:
#     {"link": "https://drive.google.com/file/d/FILE_ID/view", "name": "my_data.jsonl"}
#
#   Mounted Drive or any local path:
#     {"link": "/content/drive/MyDrive/training/my_data.jsonl", "name": "my_data.jsonl"}
#
#   Colab file picker (pops an upload dialog):
#     {"link": "upload", "name": "my_data.jsonl"}
#
# Leave as [] to use only the built-in Axiom sets above.

CUSTOM_TRAINING_FILES = [
    # Uncomment and fill in as many as you need:
    # {"link": "https://drive.google.com/file/d/YOUR_FILE_ID/view", "name": "my_finetune.jsonl"},
    # {"link": "/content/drive/MyDrive/data/custom_instructions.jsonl", "name": "custom_instructions.jsonl"},
    # {"link": "upload", "name": "uploaded_data.jsonl"},
]

# ─── derived (do not edit below) ─────────────────────────────────────────────
import re
from pathlib import Path

GGUF_DIR = OUTPUT_DIR / "gguf"
GGUF_DIR.mkdir(exist_ok=True)

if AXM_NAME is None:
    AXM_NAME = Path(GGUF_FILENAME).stem + ".axm"
AXM_OUTPUT = OUTPUT_DIR / AXM_NAME

print(f"GGUF filename:   {GGUF_FILENAME}")
print(f"AXM output:      {AXM_OUTPUT}")
print(f"Model:           {MODEL_NAME}")
print(f"Built-in sets:   {'yes (' + ', '.join(TRAINING_SETS) + ')' if INCLUDE_TRAINING_DATA else 'no'}")
print(f"Custom files:    {len(CUSTOM_TRAINING_FILES)} configured")
if GDRIVE_LINK == "PASTE_YOUR_GOOGLE_DRIVE_LINK_HERE":
    print("\n⚠  Edit GDRIVE_LINK above before running Cell 4")

In [ ]:
# Cell 4 — Download GGUF from Google Drive
import gdown, os, re, time
from pathlib import Path

if GDRIVE_LINK == "PASTE_YOUR_GOOGLE_DRIVE_LINK_HERE":
    raise ValueError("Edit GDRIVE_LINK in Cell 3 first")

gguf_dest = GGUF_DIR / GGUF_FILENAME

if gguf_dest.exists():
    gb = gguf_dest.stat().st_size / 1024**3
    print(f"✓ Already downloaded: {gguf_dest.name} ({gb:.2f} GB)")
elif GDRIVE_LINK.startswith("/") or GDRIVE_LINK.startswith("/content"):
    # Direct file path (mounted Drive)
    import shutil
    src = Path(GDRIVE_LINK)
    if not src.exists():
        raise FileNotFoundError(f"File not found: {src}")
    print(f"Copying from mounted Drive: {src.name}...")
    shutil.copy2(src, gguf_dest)
    gb = gguf_dest.stat().st_size / 1024**3
    print(f"✓ Copied: {gguf_dest.name} ({gb:.2f} GB)")
else:
    # Google Drive share link — extract file ID
    def _gdrive_id(link: str) -> str:
        m = re.search(r"/d/([a-zA-Z0-9_-]{25,})", link)
        if m:
            return m.group(1)
        m = re.search(r"[?&]id=([a-zA-Z0-9_-]{25,})", link)
        if m:
            return m.group(1)
        raise ValueError(
            f"Cannot parse Google Drive file ID from: {link}\n"
            f"Expected format: https://drive.google.com/file/d/FILE_ID/view\n"
            f"Or mount your Drive and set GDRIVE_LINK to the file path instead."
        )

    file_id = _gdrive_id(GDRIVE_LINK)
    print(f"Google Drive file ID: {file_id}")
    print(f"Downloading to {gguf_dest}...")
    t0 = time.time()
    gdown.download(id=file_id, output=str(gguf_dest), quiet=False)
    elapsed = time.time() - t0
    if not gguf_dest.exists():
        raise RuntimeError(
            "Download failed — gdown returned but file not found.\n"
            "If you get a 'Too many requests' error from Google Drive, try:\n"
            "  1. Mount your Drive (Runtime → Mount Drive) and use the file path instead\n"
            "  2. Or download the file manually and upload it to /content/"
        )
    gb = gguf_dest.stat().st_size / 1024**3
    print(f"✓ Downloaded: {gguf_dest.name} ({gb:.2f} GB) in {elapsed/60:.1f} min")

GGUF_PATH = gguf_dest
print(f"\nGGUF ready: {GGUF_PATH}")

In [ ]:
# Cell 5 — Prepare training data (built-in Axiom sets + your custom files)
import gdown, re, shutil
from pathlib import Path

CUSTOM_DIR = OUTPUT_DIR / "custom_training"
CUSTOM_DIR.mkdir(exist_ok=True)
TRAINING_FILES = []

def _gdrive_id(link: str) -> str:
    m = re.search(r"/d/([a-zA-Z0-9_-]{25,})", link)
    if m:
        return m.group(1)
    m = re.search(r"[?&]id=([a-zA-Z0-9_-]{25,})", link)
    if m:
        return m.group(1)
    raise ValueError(f"Cannot parse Google Drive file ID from: {link}")

# ── 1. Built-in Axiom training sets ──────────────────────────────────────────
if INCLUDE_TRAINING_DATA:
    print("Built-in Axiom training sets:")
    TRAINING_DIR = AXIOM_DIR / "autotrain_data"
    for name in TRAINING_SETS:
        src = TRAINING_DIR / f"{name}.jsonl"
        if not src.is_file():
            print(f"  ⚠ Not found: {src} — skipping")
            continue
        count   = sum(1 for _ in open(src))
        size_kb = src.stat().st_size / 1024
        print(f"  ✓ {name}.jsonl  ({count:,} examples, {size_kb:.0f} KB)")
        TRAINING_FILES.append(src)
else:
    print("Built-in sets: skipped (INCLUDE_TRAINING_DATA=False)")

# ── 2. Custom JSONL files ─────────────────────────────────────────────────────
if CUSTOM_TRAINING_FILES:
    print(f"\nCustom training files ({len(CUSTOM_TRAINING_FILES)} configured):")
    for entry in CUSTOM_TRAINING_FILES:
        link = entry["link"]
        name = entry["name"]
        dest = CUSTOM_DIR / name

        if dest.exists():
            count   = sum(1 for _ in open(dest))
            size_kb = dest.stat().st_size / 1024
            print(f"  ✓ {name}  ({count:,} examples, {size_kb:.0f} KB)  [already downloaded]")
            TRAINING_FILES.append(dest)
            continue

        if link == "upload":
            # Colab file picker
            try:
                from google.colab import files as colab_files
                print(f"  Upload dialog → select '{name}'")
                uploaded = colab_files.upload()
                for fname, data in uploaded.items():
                    dest.write_bytes(data)
                    print(f"  ✓ uploaded as {name}")
                    break
            except ImportError:
                print(f"  ⚠ upload mode only works in Colab — skipping {name}")
                continue

        elif link.startswith("/") or link.startswith("/content"):
            # Local or mounted Drive path
            src = Path(link)
            if not src.exists():
                print(f"  ✗ Not found: {src}")
                continue
            shutil.copy2(src, dest)
            print(f"  ✓ copied: {name}")

        else:
            # Google Drive share link
            try:
                file_id = _gdrive_id(link)
            except ValueError as e:
                print(f"  ✗ {name}: {e}")
                continue
            print(f"  Downloading {name} from Google Drive...")
            gdown.download(id=file_id, output=str(dest), quiet=False)
            if not dest.exists():
                print(f"  ✗ Download failed for {name}")
                continue
            print(f"  ✓ {name}")

        if dest.exists():
            count   = sum(1 for _ in open(dest))
            size_kb = dest.stat().st_size / 1024
            print(f"    {count:,} examples, {size_kb:.0f} KB")
            TRAINING_FILES.append(dest)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
if TRAINING_FILES:
    total = sum(sum(1 for _ in open(f)) for f in TRAINING_FILES)
    print(f"Total: {len(TRAINING_FILES)} files, {total:,} training examples to bundle")
    for f in TRAINING_FILES:
        count = sum(1 for _ in open(f))
        print(f"  · {f.name}  ({count:,})")
else:
    total = 0
    print("No training data — weights only.")

TOTAL_TRAINING_EXAMPLES = total

In [ ]:
# Cell 6 — Pack GGUF + training data → signed .axm
import os, re, shutil, tempfile, time
from pathlib import Path

sys.path.insert(0, str(AXIOM_DIR))
from axiom_axm import AXMContainer, FORMAT_VERSION

if AXM_OUTPUT.exists():
    gb = AXM_OUTPUT.stat().st_size / 1024**3
    print(f"✓ Already packed: {AXM_OUTPUT.name} ({gb:.3f} GB)")
    print("  Delete the file and re-run this cell to repack.")
else:
    # Infer quant type from filename
    _GGUF_BPW = {
        "Q4_K_M": 4.85, "Q4_K_S": 4.6, "Q4_0": 4.5, "Q4_1": 5.0,
        "Q5_K_M": 5.70, "Q5_K_S": 5.5, "Q5_0": 5.5, "Q5_1": 6.0,
        "Q6_K":   6.6,  "Q8_0":   8.5, "Q8_1": 9.0,
        "Q3_K_L": 4.0,  "Q3_K_M": 3.9, "Q3_K_S": 3.5, "Q2_K": 2.6,
        "IQ4_XS": 4.4,  "IQ4_NL": 4.5, "IQ3_S": 3.5, "IQ2_XS": 2.3,
        "F16":   16.0,  "BF16": 16.0,  "F32": 32.0,
    }
    name_upper = GGUF_PATH.name.upper().replace("_", "-")
    quant_name = "Q4_K_M"  # safe default
    bpw        = 4.85
    for q in sorted(_GGUF_BPW, key=len, reverse=True):
        if q.replace("_", "-") in name_upper:
            quant_name = q
            bpw = _GGUF_BPW[q]
            break

    gguf_gb = GGUF_PATH.stat().st_size / 1024**3
    print(f"Packing: {GGUF_PATH.name}")
    print(f"  quant:   {quant_name}  ({bpw:.2f} bpw)")
    print(f"  size:    {gguf_gb:.3f} GB")
    print(f"  model:   {MODEL_NAME}")
    if TRAINING_FILES:
        print(f"  training: {TOTAL_TRAINING_EXAMPLES:,} examples from {len(TRAINING_FILES)} files")

    t0 = time.monotonic()
    with tempfile.TemporaryDirectory(prefix="axm_qwen_") as tmp:
        weights_dir = Path(tmp) / "weights"
        weights_dir.mkdir()

        # ── 1. Copy GGUF weights ───────────────────────────────────────────────
        print("  Copying GGUF...")
        shutil.copy2(GGUF_PATH, weights_dir / "model.gguf")

        # ── 2. Bundle training data ────────────────────────────────────────────
        td_meta = []
        if TRAINING_FILES:
            print("  Bundling training data...")
            td_dir = weights_dir / "training_data"
            td_dir.mkdir()
            for src in TRAINING_FILES:
                dest = td_dir / src.name
                shutil.copy2(src, dest)
                count = sum(1 for _ in open(dest))
                td_meta.append({"file": src.name, "examples": count})
                print(f"    + {src.name}  ({count:,} examples)")

        # ── 3. Build container spec ────────────────────────────────────────────
        spec = {
            "format_version":  FORMAT_VERSION,
            "core_logic":      MODEL_NAME.split("/")[-1],
            "quant_map": {
                "scheme":  "gguf",
                "format":  quant_name,
                "bpw":     bpw,
                "note":    f"Pre-quantized GGUF ({quant_name}) packed into AXM for governed edge deployment",
            },
            "hardware_map":   "llama.cpp",
            "safety_proofs":  True,
            "core": {
                "name":   MODEL_NAME,
                "format": "gguf",
                "quant":  quant_name,
            },
            "training_data": {
                "included": bool(TRAINING_FILES),
                "files":    td_meta,
                "total_examples": TOTAL_TRAINING_EXAMPLES,
                "source":   "orivael-dev/axiom autotrain_data/",
            },
        }

        # ── 4. Pack and sign ───────────────────────────────────────────────────
        print("  Signing and writing .axm...")
        container = AXMContainer.pack(
            weights_dir,
            output_path=AXM_OUTPUT,
            spec=spec,
            compresslevel=1,  # GGUF already compressed
        )

    elapsed = time.monotonic() - t0
    axm_gb  = AXM_OUTPUT.stat().st_size / 1024**3

    print(f"\n✓ Done in {elapsed:.1f}s")
    print(f"  .axm:        {AXM_OUTPUT}  ({axm_gb:.3f} GB)")
    print(f"  fingerprint: {container.fingerprint}")
    print(f"  proofs:      {len(container.proofs)}")

    CONTAINER_FINGERPRINT = container.fingerprint
    CONTAINER_PROOFS      = len(container.proofs)

In [ ]:
# Cell 7 — Verify HMAC proof chain
import json, subprocess, sys

AXM_CLI = AXIOM_DIR / "axm_cli.py"

print(f"Verifying: {AXM_OUTPUT.name}")
result = subprocess.run(
    [sys.executable, str(AXM_CLI), "verify", str(AXM_OUTPUT)],
    cwd=str(AXIOM_DIR), capture_output=True, text=True,
)

try:
    data = json.loads(result.stdout)
except Exception:
    data = {"verified": False, "error": result.stdout + result.stderr}

ok     = data.get("verified", False)
fp     = data.get("fingerprint", "?")
proofs = data.get("proofs_checked", "?")

if ok:
    print(f"✓ VERIFIED")
    print(f"  fingerprint:  {fp}")
    print(f"  proofs:       {proofs}")
    print(f"\nThis fingerprint is the public commitment to these exact weights.")
    print(f"Anyone with the AXIOM_MASTER_KEY can verify this AXM is untampered.")
else:
    print(f"✗ VERIFICATION FAILED")
    print(f"  {data.get('error', 'unknown error')}")
    print("\nPossible causes:")
    print("  - AXIOM_MASTER_KEY was regenerated (re-run Cell 2 to restore key, then repack)")
    print("  - AXM file was modified after packing")

In [ ]:
# Cell 8 — Summary + download + inference commands
import json
from pathlib import Path

print("=" * 65)
print("AXM PACK COMPLETE")
print("=" * 65)

axm_gb   = AXM_OUTPUT.stat().st_size / 1024**3
gguf_gb  = GGUF_PATH.stat().st_size / 1024**3
key_hex  = KEY_FILE.read_text().strip() if KEY_FILE.is_file() else "(from environment)"

print(f"  Model:        {MODEL_NAME}")
print(f"  Fingerprint:  {CONTAINER_FINGERPRINT}")
print(f"  Proofs:       {CONTAINER_PROOFS}")
print(f"  .axm size:    {axm_gb:.3f} GB  →  {AXM_OUTPUT.name}")
print(f"  .gguf size:   {gguf_gb:.3f} GB")
if TRAINING_FILES:
    print(f"  Training:     {TOTAL_TRAINING_EXAMPLES:,} examples bundled inside AXM")
print()
print("AXIOM_MASTER_KEY (save this to verify later):")
print(f"  {key_hex}")

# Save results JSON
results = {
    "model":         MODEL_NAME,
    "fingerprint":   CONTAINER_FINGERPRINT,
    "proofs":        CONTAINER_PROOFS,
    "quant":         quant_name,
    "bpw":           bpw,
    "axm_path":      str(AXM_OUTPUT),
    "gguf_path":     str(GGUF_PATH),
    "training_examples": TOTAL_TRAINING_EXAMPLES,
    "training_files": [str(f) for f in TRAINING_FILES],
}
results_file = OUTPUT_DIR / "results.json"
results_file.write_text(json.dumps(results, indent=2))
print(f"\nResults saved: {results_file}")

# Download to local machine
print("\n" + "-" * 65)
print("DOWNLOAD to your computer")
print("-" * 65)
try:
    from google.colab import files
    print("Downloading .axm to your local machine...")
    files.download(str(AXM_OUTPUT))
    print(f"✓ {AXM_OUTPUT.name}")
    print()
    print("Downloading results.json...")
    files.download(str(results_file))
except ImportError:
    print(f"  Not in Colab — copy from: {AXM_OUTPUT}")

# Inference commands
print("\n" + "-" * 65)
print("INFERENCE with llama.cpp")
print("-" * 65)
print(f"""# Interactive chat
{LLAMA_BIN} \\\n  -m '{GGUF_PATH}' \\\n  --ngl 99 \\\n  --ctx-size 4096 \\\n  -p '<|im_start|>user\\nWrite a Python function to reverse a string<|im_end|>\\n<|im_start|>assistant\\n' \\\n  -n 256 --temp 0.7
# Single prompt (non-interactive)
{LLAMA_BIN} -m '{GGUF_PATH}' --ngl 99 -p 'Hello' -n 64 -e""")

# Verify command (for edge deployment)
print("\n" + "-" * 65)
print("VERIFY on another machine")
print("-" * 65)
print(f"""export AXIOM_MASTER_KEY={key_hex}
python3 /path/to/axiom/axm_cli.py verify {AXM_OUTPUT.name}
# Expected: fingerprint={CONTAINER_FINGERPRINT}""")